In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('Gas_Sensors_Measurements_value.csv')

In [ ]:
df

In [ ]:
# get unique values of the column gas
df['Gas'].unique()
# count
df['Gas'].value_counts()

In [ ]:
data = df.drop(['Serial Number','MQ2', 'MQ3', 'MQ5', 'MQ6', 'MQ8', 'Corresponding Image Name'], axis=1)

In [ ]:
# standart scaller
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(data.drop('Gas', axis=1))
scaled_features = scaler.transform(data.drop('Gas', axis=1))
df_feat = pd.DataFrame(scaled_features, columns=data.columns[:-1])
df_feat.head()


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(scaled_features, data['Gas'], test_size=0.30)


In [ ]:
# y_train, y_test to labelencoder
# from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder()
# y_train = le.fit_transform(y_train)
# y_test = le.transform(y_test)

# show le classes name
# le.classes_


In [ ]:
# model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import BatchNormalization

model = Sequential()
model.add(Dense(units=64,activation='relu'))
model.add(Dense(units=32,activation='relu'))

# BatchNormalization
model.add(BatchNormalization())

model.add(Dense(units=4,activation='sigmoid'))

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

history = model.fit(x=X_train, y=y_train, epochs=25, validation_data=(X_test, y_test), verbose=1)


In [ ]:
# hitory plot
import matplotlib.pyplot as plt
%matplotlib inline
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train','validation'], loc='upper left')
plt.show()


In [ ]:
model.save('gas.h5')

In [ ]:
import tensorflow as tf
model = tf.keras.models.load_model('gas.h5')

In [ ]:
df = pd.read_csv('gas_testing.csv')
# model.predict()
data_y = df['Gas']
data = df.drop(['Serial Number','MQ2', 'MQ3', 'MQ5', 'MQ6', 'MQ8', 'Corresponding Image Name', 'Gas'], axis=1)

In [ ]:
data

In [ ]:
scaled_features = scaler.transform(data)
df_feat = pd.DataFrame(scaled_features, columns=data.columns)
df_feat.head()

In [ ]:

pred = model.predict(df_feat)
kelas = ['No Gas', 'Perfume', 'Smoke', 'Mixture']
hasil = []
for i in pred:
  print(kelas[i.argmax()])
  hasil.append(i.argmax())
  



In [ ]:
hasil

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
cm = confusion_matrix(data_y, hasil)
cm

In [ ]:
import matplotlib.pyplot as plt
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Gas', 'Perfume', 'Smoke', 'Mixture'])
disp.plot()
plt.show()

In [ ]:
import numpy as np
FP = cm.sum(axis=0) - np.diag(cm)  
FN = cm.sum(axis=1) - np.diag(cm)
TP = np.diag(cm)
TN = cm.sum() - (FP + FN + TP)

# Sensitivity, hit rate, recall, or true positive rate
TPR = TP/(TP+FN)
# Specificity or true negative rate
TNR = TN/(TN+FP) 
# Precision or positive predictive value
PPV = TP/(TP+FP)
# Negative predictive value
NPV = TN/(TN+FN)
# Fall out or false positive rate
FPR = FP/(FP+TN)
# False negative rate
FNR = FN/(TP+FN)
# False discovery rate
FDR = FP/(TP+FP)

# Overall accuracy
ACC = (TP+TN)/(TP+FP+FN+TN)

In [ ]:
print("Specificity or true negative rate : \n" , TNR)
print("Sensitivity, hit rate, recall, or true positive rate : \n" , TPR)
print("Fall out or false positive rate : \n" , FPR)
print("False negative rate : \n" , FNR)
print("Precision or positive predictive value : \n" , PPV)
print("Negative predictive value : \n" , NPV)
print("False discovery rate : \n" , FDR)
print("Overall accuracy : \n" , ACC)
